In [ ]:
import os
path = "/..."
os.chdir(path + "/llm-introspection-main")

In [ ]:
%pip install textattack
import textattack
from huggingface_hub import login
login(token="...")

In [ ]:
import sys
sys.path.append(path + "/wrapper")
from wrapper import TextAttackWrapper

In [ ]:
from textattack.constraints import Constraint
from textattack.shared import AttackedText

class OnlyAfterPeriodConstraint(Constraint):
    """
    Allows modifications only after the first period ('.')
    in the original input.
    """
    def __init__(self):
        super().__init__(compare_against_original=True)

    def _check_constraint(self, transformed_text: AttackedText, current_text: AttackedText) -> bool:
        orig_text = current_text.text
        period_index = orig_text.find('.')

        if period_index == -1:
            return False

        word_starts = []
        idx = 0
        for word in current_text.words:
            idx = orig_text.find(word, idx)
            if idx == -1:
                break
            word_starts.append(idx)
            idx += len(word)

        modified_indices = transformed_text.attack_attrs.get("modified_indices", [])
        if not modified_indices:
            return True

        return all(word_starts[i] > period_index for i in modified_indices)

Load the Dataset

In [ ]:
os.chdir(path + '/eraserbenchmark-master')
from rationale_benchmark.utils import load_documents, annotations_from_jsonl
esnli_data_root = path + '/eraserbenchmark-master/data/esnli'
esnli_documents = load_documents(esnli_data_root)
esnli = annotations_from_jsonl(os.path.join(esnli_data_root, 'test.jsonl'))
esnli = [inst for inst in esnli if inst.classification in ('entailment', 'contradiction')]

# Llama

In [ ]:
from transformers import LlamaForCausalLM, AutoTokenizer
import torch

model_name = "meta-llama/Llama-3.2-3B-Instruct"
model = LlamaForCausalLM.from_pretrained(model_name,device_map="cuda",torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True,padding_side='left')
tokenizer.pad_token = tokenizer.eos_token

In [34]:
wrapper = TextAttackWrapper(model_family="llama", model=model, tokenizer=tokenizer, task='entailment', batch_size=64)

In [ ]:
import pandas as pd
from textattack.datasets import Dataset

os.chdir(path + '/eraserbenchmark-master/esnli_dataset_builder/my_dataset')
df = pd.read_csv("test.csv")

data = []
for row in df.iterrows():
    hypothesis = row[1][0].replace('.','')
    premise = row[1][1]
    text = hypothesis+'.'+premise 
    res = wrapper([text])
    data.append(([text], int(res[0][1] > res[0][0])))

marked_dataset = Dataset(data)
num_examples = len(marked_dataset)

In [ ]:
from textattack.attack_recipes import TextFoolerJin2019
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:32'
os.chdir(path)

attack_recipe = TextFoolerJin2019
log_filename = f"esnli_attacks/Llama-3.2-3B-Instruct_{attack_recipe.__name__}.csv"
attack_args = textattack.AttackArgs(num_examples=num_examples, log_to_csv=log_filename, disable_stdout= False, parallel=False)
attack = attack_recipe.build(wrapper)
attack.constraints.append(OnlyAfterPeriodConstraint())
attacker = textattack.Attacker(attack, marked_dataset, attack_args)
attacker.attack_dataset()
del attacker
torch.cuda.empty_cache()

# Qwen

In [51]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name,device_map="cuda",torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True,padding_side='left')
tokenizer.pad_token = tokenizer.eos_token

wrapper = TextAttackWrapper(model_family="qwen", model=model, tokenizer=tokenizer, task='entailment', batch_size=64)

[Succeeded / Failed / Skipped / Total] 11 / 6 / 0 / 17:   0%|          | 17/6598 [05:07<33:04:27, 18.09s/it]
textattack: CSVLogger exiting without calling flush().


In [ ]:
import pandas as pd
from textattack.datasets import Dataset

os.chdir(path + '/eraserbenchmark-master/esnli_dataset_builder/my_dataset')
df = pd.read_csv("test.csv")

data = []

for row in df.iterrows():
    hypothesis = row[1][0].replace('.','')
    premise = row[1][1]
    text = hypothesis+'.'+premise 

    res = wrapper([text])
    data.append(([text], int(res[0][1] > res[0][0])))


marked_dataset = Dataset(data)
num_examples = len(marked_dataset)

In [ ]:
from textattack.attack_recipes import TextFoolerJin2019
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:32'
os.chdir(path)

attack_recipe = TextFoolerJin2019
log_filename = f"esnli_attacks/Qwen-1.5B_{attack_recipe.__name__}.csv"
attack_args = textattack.AttackArgs(num_examples=num_examples, log_to_csv=log_filename, disable_stdout=True, parallel=False)
attack = attack_recipe.build(wrapper)
attack.constraints.append(OnlyAfterPeriodConstraint())
attacker = textattack.Attacker(attack, marked_dataset, attack_args)
attacker.attack_dataset()
del attacker
torch.cuda.empty_cache()

textattack: Unknown if model of class <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
textattack: Logging to CSV at path esnli_attacks/Qwen-1.5B_TextFoolerJin2019.csv


Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (2): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.840845057
        (window_size):  15
        (skip_text_shorter_than_window):  True
        (compare_against_original):  False
      )
    (3): OnlyAfterPeriodConstraint(
        (compare_against_original):  True
      )
    (4): RepeatModification
    (5): StopwordModification
    (6): InputCo

/venv/main/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/venv/main/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
[Succeeded / Failed / Skipped / Total] 22 / 14 / 0 / 36:   1%|          | 36/6598 [00:40<2:01:36,  1.11s/it]